In [1]:
import transformers
import datasets
import accelerate
import evaluate
import tqdm
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("Accelerate version:", accelerate.__version__)
print("Evaluate version:", evaluate.__version__)

d:\anaconda\envs\huggingface_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 4.51.3
Datasets version: 3.5.0
Accelerate version: 1.5.2
Evaluate version: 0.4.3


In [2]:
from datasets import load_dataset

# 加载 WMT19 英语-法语数据集
dataset = load_dataset("wmt19", "zh-en")

Using the latest cached version of the dataset since wmt19 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'zh-en' at C:\Users\wyx\.cache\huggingface\datasets\wmt19\zh-en\0.0.0\c8a87a86736e7bf10c74c22e9033eba9146774e9 (last modified on Fri Apr 18 12:46:28 2025).


In [ ]:
train_val_split=dataset["train"].train_test_split(test_size=0.01,seed=42)
#将原有的数据集划分，使用小的数据集进行实验，使用原训练集的1%的数据进行实验
small_train_dataset=train_val_split['test']
small_validation_dataset=dataset['validation']
print(small_train_dataset)
print(small_validation_dataset)

Dataset({
    features: ['translation'],
    num_rows: 259846
})
Dataset({
    features: ['translation'],
    num_rows: 3981
})


In [4]:
import os
current_dir=os.getcwd()
print(current_dir)

d:\工作\test


In [8]:
from transformers import AutoTokenizer
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizerFast
from datasets import load_dataset
#使用bert的wordpiece分词方法
#源语言与目标语言
source_lang='zh'
target_lang='en'

zh_train_file="zh_small_tokenizer.txt"
en_train_file="en_small_tokenizer.txt"

train_dataset=small_train_dataset
zh_sentences=[item['translation'][source_lang] for item in train_dataset]
en_sentences=[item['translation'][target_lang] for item in train_dataset]

#训练分词器所需的句子
tokenizer_need_samples=50000
zh_tokenizer_sentences=zh_sentences[:tokenizer_need_samples]
en_tokenizer_sentences=en_sentences[:tokenizer_need_samples]

#保存分词所需的句子
with open(zh_train_file,'w',encoding='utf-8') as f:
    for sentence in zh_tokenizer_sentences:
        f.write(sentence+'\n')
with open(en_train_file,'w',encoding='utf-8') as f:
    for sentence in en_tokenizer_sentences:
        f.write(sentence+'\n')

In [9]:
#一些参数设置
vocab_size = 30000
min_frequency = 2
unk_token = "[UNK]"
sep_token = "[SEP]"
cls_token = "[CLS]"
pad_token = "[PAD]"
mask_token = "[MASK]"
special_tokens = [unk_token, sep_token, cls_token, pad_token, mask_token]

In [ ]:
# 训练中文分词器
zh_tokenizer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=True,
    lowercase=False
)

zh_tokenizer.train(
    files=[zh_train_file],
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    show_progress=True,
    special_tokens=special_tokens
)
zh_tokenizer.save_model("zh_wordpiece_tokenizer")

['zh_wordpiece_tokenizer\\vocab.txt']

In [16]:
# 训练英文分词器
en_tokenizer = BertWordPieceTokenizer(
    clean_text=True,
    handle_chinese_chars=False,
    strip_accents=True,
    lowercase=True
)

en_tokenizer.train(
    files=[en_train_file],
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    show_progress=True,
    special_tokens=special_tokens
)
en_tokenizer.save_model("en_wordpiece_tokenizer")

['en_wordpiece_tokenizer\\vocab.txt']

In [19]:
# 清理临时文件
os.remove(zh_train_file)
os.remove(en_train_file)

print("WordPiece 分词器训练完成并保存。")

WordPiece 分词器训练完成并保存。


In [17]:
#将保存好的分词器加载出来
from transformers import BertTokenizerFast

source_lang = "zh"
target_lang = "en"
zh_load_tokenizer = BertTokenizerFast.from_pretrained("zh_wordpiece_tokenizer")
en_load_tokenizer = BertTokenizerFast.from_pretrained("en_wordpiece_tokenizer")

In [18]:
print(f"中文分词器的 [UNK] token: {zh_load_tokenizer.unk_token}")
print(f"英文分词器的 [UNK] token: {en_load_tokenizer.unk_token}")
print(f"中文分词器词汇表大小: {zh_load_tokenizer.vocab_size}")
print(f"英文分词器词汇表大小: {en_load_tokenizer.vocab_size}")

中文分词器的 [UNK] token: [UNK]
英文分词器的 [UNK] token: [UNK]
中文分词器词汇表大小: 4684
英文分词器词汇表大小: 29785
